# BM25 top-10 000 по каждому полю → объединение → ближайшие 50

Бейзлайн из трёх источников: `search_query → item_title_raw`,
`search_query → item_description_raw` и `search_infm_params_text → item_infm_params_text`.
Для каждого источника забираем top-10 000, объединяем объявления по item_id и из union
выбираем 50 ближайших к центру локации. BM25 использует лемматизацию и статистики полного
корпуса; индекс строится один раз на источник, затем скорит val и test.

Сплит — `split_queries(seed=42)`, 500 контекстов отдельно на val и test. Центры считаются
только по train: item_location_id, с fallback по search_location_id. В конце показываем
Recall@50 union до геосортировки, oracle Recall@50 пула и Recall@50 после сортировки.

In [ ]:
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/split.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Откройте ноутбук из репозитория или задайте ROOT.")
DATA_PATH = ROOT / "data/train.parquet"
N_QUERIES = 500  # None — все запросы части
SEED = 42
N_RETRIEVAL = 10_000
K = 50
BATCH_SIZE = 16
LEMMATIZE = True
ARTIFACTS = ROOT / "artifacts/bm25_top10k_distance"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
assert N_RETRIEVAL >= K

In [ ]:
import gc
import hashlib
import json
from time import perf_counter
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display
from src.split import split_queries
from src.geo import location_centers
from src.retrievers.bm25 import BM25Retriever
from src.retrievers.hybrid import HybridRetriever

QUERY_COLS = ["search_query", "search_location_id", "search_is_delivery_search", "search_infm_params_text", "search_category"]
GEO_COLS = ["item_location_id", "item_latitude", "item_longitude"]
data = pl.read_parquet(DATA_PATH, columns=QUERY_COLS + ["item_id"] + GEO_COLS)
data = data.with_columns(
    pl.Series("split", split_queries(data["search_query"], seed=SEED)),
    pl.col("item_latitude", "item_longitude").cast(pl.Float64),
)
items = pd.DataFrame(data.select("item_id", *GEO_COLS).unique(subset="item_id", keep="last", maintain_order=True)
                     .sort("item_id").to_dict(as_series=False))
train_rows = data.filter(pl.col("split") == "train").select("search_location_id", "item_id", *GEO_COLS)
centers = location_centers(pd.DataFrame(train_rows.to_dict(as_series=False)))
queries, targets = {}, {}
for split in ("val", "test"):
    part = data.filter(pl.col("split") == split).group_by(QUERY_COLS).agg(pl.col("item_id").unique().alias("targets")).sort(QUERY_COLS)
    if N_QUERIES is not None:
        part = part.sample(n=min(N_QUERIES, part.height), seed=SEED, shuffle=True)
    queries[split] = pd.DataFrame(part.select(QUERY_COLS).to_dict(as_series=False)).join(centers, on="search_location_id")
    targets[split] = [set(ids) for ids in part["targets"].to_list()]
    assert set().union(*targets[split]).issubset(set(items.item_id))
    print(split, len(part), "контекстов; центры:", queries[split].center_source.value_counts(dropna=False).to_dict())
del data, train_rows, part
gc.collect()

## Однократный top-10 000 и кеширование

Каждый BM25 индексируем один раз, выдачи сохраняем небольшими parquet-батчами.
Повторный запуск с теми же данными, запросами и настройками читает кеш. При лемматизации
ключ кеша меняется автоматически через параметры источника.

In [ ]:
sources = {
    "bm25_title": {"item_text_column": "item_title_raw", "query_text_column": "search_query"},
    "bm25_description": {"item_text_column": "item_description_raw", "query_text_column": "search_query"},
    "bm25_infm": {"item_text_column": "item_infm_params_text", "query_text_column": "search_infm_params_text"},
}
stat = DATA_PATH.stat()
query_hash = {split: hashlib.sha256(pd.util.hash_pandas_object(q, index=True).to_numpy().tobytes()).hexdigest()
              for split, q in queries.items()}
cache_dirs = {}
for name, params in sources.items():
    signature = {"source": {**params, "lemmatize": LEMMATIZE, "k1": 1.5, "b": .75},
                 "queries": query_hash, "n": N_RETRIEVAL, "batch": BATCH_SIZE,
                 "data": [str(DATA_PATH.resolve()), stat.st_size, stat.st_mtime_ns],
                 "code": hashlib.sha256((ROOT / "src/retrievers/bm25.py").read_bytes()).hexdigest()}
    digest = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:16]
    cache_dirs[name] = ARTIFACTS / "candidates" / name / digest
    cache_dirs[name].mkdir(parents=True, exist_ok=True)
    (cache_dirs[name] / "manifest.json").write_text(json.dumps(signature, ensure_ascii=False, indent=2))

def save_candidates(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp.parquet")
    pl.DataFrame(frame.to_dict(orient="list"), schema={"query_index": pl.Int64, "item_id": pl.String,
                                                        "rank": pl.Int64, "score": pl.Float64}).write_parquet(temporary)
    temporary.replace(path)

for name, params in sources.items():
    jobs = [(split, start) for split, q in queries.items() for start in range(0, len(q), BATCH_SIZE)]
    pending = [(split, start) for split, start in jobs
               if not (cache_dirs[name] / split / f"{start:07d}.parquet").is_file()]
    if not pending:
        print(name, "— готовые выдачи в кеше")
        continue
    started = perf_counter()
    scan = pl.scan_parquet(DATA_PATH).select("item_id", params["item_text_column"])
    frame = scan.unique(subset="item_id", keep="last", maintain_order=True).sort("item_id").collect()
    corpus = pd.DataFrame(frame.to_dict(as_series=False))
    model = BM25Retriever(**params, lemmatize=LEMMATIZE, batch_size=BATCH_SIZE).fit(corpus)
    del frame, corpus
    gc.collect()
    print(name, "— индекс построен за", round(perf_counter() - started, 1), "с")
    for i, (split, start) in enumerate(pending, 1):
        batch = queries[split].iloc[start:start + BATCH_SIZE]
        result = model.retrieve(batch, k=N_RETRIEVAL)
        save_candidates(result, cache_dirs[name] / split / f"{start:07d}.parquet")
        if i % 10 == 0 or i == len(pending):
            print(name, f"{i}/{len(pending)} батчей; {perf_counter() - started:.1f} с")
    del model, result
    gc.collect()

## Union и сортировка по расстоянию

Расстояние считается до центра, построенного выше. Неизвестные координаты идут последними.
Для полноты union и потолка сортировки считаем метрики отдельно от итогового top-50.

In [ ]:
hybrid = HybridRetriever(n_candidates=N_RETRIEVAL, distance_only=True).fit(items)
rows = []
for split, q in queries.items():
    sums = np.zeros(4)
    for start in range(0, len(q), BATCH_SIZE):
        batch = q.iloc[start:start + BATCH_SIZE]
        relevant = targets[split][start:start + len(batch)]
        results = {
            name: pd.DataFrame(pl.read_parquet(cache_dirs[name] / split / f"{start:07d}.parquet").to_dict(as_series=False))
            for name in sources
        }
        pool = hybrid.combine(batch, results)
        retrieved = hybrid.rerank(pool, k=K)
        found_pool = pool.groupby("query_index").item_id.agg(set).to_dict()
        found_top = retrieved.groupby("query_index").item_id.agg(set).to_dict()
        denominator = np.array([len(ids) for ids in relevant])
        pool_hits = np.array([len(ids & found_pool.get(i, set())) for i, ids in enumerate(relevant)])
        top_hits = np.array([len(ids & found_top.get(i, set())) for i, ids in enumerate(relevant)])
        sums += [np.sum(pool_hits / denominator), np.sum(np.minimum(pool_hits, K) / denominator),
                 np.sum(top_hits / denominator), len(pool)]
    rows.append({"split": split, "queries": len(q), "pool_recall": sums[0] / len(q),
                 "oracle_recall@50": sums[1] / len(q), "distance_recall@50": sums[2] / len(q),
                 "mean_union_size": sums[3] / len(q)})
comparison = pd.DataFrame(rows)
comparison.to_csv(ARTIFACTS / "comparison.csv", index=False)
display(comparison)
print("Candidate cache:", ARTIFACTS / "candidates")